# S5.G3-2 · G3 五臂对比 FedAds：联邦臂 A（隐语 SGB）

> 隐语 SGB 单进程两方模拟（OU 同态加密），超参取自集中式代理 L4 的调参结果（S6.P3 freeze A-4）。耗时只作臂间相对比较，不作部署成本。

In [1]:
import hashlib
import json
import os
import pathlib
import subprocess
import sys
import time

import numpy as np
import pandas as pd
import yaml

HASH_PREFIX_LEN = 12

REPO_ROOT = pathlib.Path.cwd()
while not (REPO_ROOT / "AGENTS.md").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

CONFIG_PATH = REPO_ROOT / "modules/m5_modeling/configs/s5_g3_fedads_run.yaml"
run = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
PRE_PATH, ADD_PATH = REPO_ROOT / run["preregistration"], REPO_ROOT / run["addendum"]
pre = yaml.safe_load(PRE_PATH.read_text(encoding="utf-8"))
add = yaml.safe_load(ADD_PATH.read_text(encoding="utf-8"))
ad, fa = pre["freeze"]["arm_details"], add["arm_details"]


def sha(path, n=HASH_PREFIX_LEN):
    return hashlib.sha256(pathlib.Path(path).read_bytes()).hexdigest()[:n]


git_sha = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True, cwd=REPO_ROOT).strip()
dirty = subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True,
                       cwd=REPO_ROOT).stdout.strip() != ""
print("config:", CONFIG_PATH.relative_to(REPO_ROOT), "sha256:" + sha(CONFIG_PATH))
print("preregistration:", PRE_PATH.relative_to(REPO_ROOT), "sha256:" + sha(PRE_PATH), "| status:", pre["status"])
print("addendum:", ADD_PATH.relative_to(REPO_ROOT), "sha256:" + sha(ADD_PATH), "| status:", add["status"])
print("seed:", run["seed"], "| seeds:", run["seeds"])
print("git:", git_sha + ("（工作区含本步未提交改动）" if dirty else ""))
print("step:", run["step_id"], "|", run["purpose"])
mismatch = [p for p, h in add["code_sha256"].items() if hashlib.sha256((REPO_ROOT / p).read_bytes()).hexdigest() != h]
print("冻结代码指纹核对：", "一致" if not mismatch else "不一致 " + str(mismatch))
assert pre["status"] == "frozen" and add["status"] == "frozen" and not mismatch
assert sha(PRE_PATH, None) == add["parent_sha256"]
PRIV = pathlib.Path(os.environ[run["private_root_env"]])
SEED0 = run["seeds"][0]
MB, NJ = ad["xgboost_max_bin"], ad["n_jobs"]

config: modules/m5_modeling/configs/s5_g3_fedads_run.yaml sha256:56f1fdbd6540
preregistration: modules/m6_evaluation/configs/s6_p3_g3_preregistration.yaml sha256:c0a4bbfc9709 | status: frozen
addendum: modules/m6_evaluation/configs/s6_p4_fedads_addendum.yaml sha256:462b0a74e10b | status: frozen
seed: 11 | seeds: [11, 22, 33, 44, 55, 66]
git: 540d5b9（工作区含本步未提交改动）
step: S5.G3 | 按冻结的 S6.P3 与 S6.P4 在 FedAds 上训练 L0、L1（k=5/10/20）、B_T1、A（SGB）、集中式代理 L4 与阴性对照 NC，并做配对比较
冻结代码指纹核对： 一致


In [2]:
from modules.m5_modeling.components import g3_sgb as SG
from sklearn.metrics import roc_auc_score

arr = np.load(PRIV / run["arrays_file"])
TUNING = json.loads((PRIV / run["tuning_file"]).read_text(encoding="utf-8"))
params = SG.sgb_params(TUNING["A"]["best"], ad["sgb_sketch_eps"], SEED0)
print("A 的 SGB 参数（来自 L4 集中式代理调参）：", params)
sets = {n: (arr[n + "__xg"], arr[n + "__xh"]) for n in ("inner_val", "dev_eval")}
scores, timing = SG.fit_and_score(arr["train__xg"], arr["train__xh"], arr["train__y"], sets, params, ad["sgb_he_schema"])
np.savez_compressed(PRIV / run["sgb_scores_file"], **scores)
print("SGB 训练 %.0f 秒，打分 %.0f 秒" % (timing["fit_seconds"], timing["predict_seconds"]))
print("内层验证集 AUROC：SGB %.4f ｜ L4 集中式代理调参时 %.4f"
      % (roc_auc_score(arr["inner_val__y"], scores["inner_val"]), TUNING["L4"]["best_val_auc"]))
(PRIV / run["timing_file"]).write_text(json.dumps(timing), encoding="utf-8")

A 的 SGB 参数（来自 L4 集中式代理调参）： {'num_boost_round': 150, 'max_depth': 3, 'learning_rate': 0.2, 'reg_lambda': 1.0, 'sketch_eps': 0.03, 'objective': 'logistic', 'seed': 11}


[2026-10-27 17:02:13.391] [info] [bigint_spi.cc:79] The default library used for BigInt operations is openssl


[2026-10-27 17:02:22.609] [info] [thread_pool.cc:30] Create a fixed thread pool with size 1


[已折叠 150 行相同的 SecretFlow 运行警告：WARNING:root:no effective dump_function provided.]


SGB 训练 2343 秒，打分 8 秒
内层验证集 AUROC：SGB 0.6332 ｜ L4 集中式代理调参时 0.6351


69

**读法**：SGB 训练 2,343 秒（约 39 分钟，15 万行、深 3、150 轮）。内层验证集 AUROC 0.6332，与集中式代理 L4 调参时的 0.6351 相差 0.002，符合 SecureBoost 类协议在分桶相同时与集中式近似无损的预期；开发评估集上的无损性核对见 S5.G3-3。150 行相同的 SecretFlow 运行警告已折叠为一行说明。